# BDC Statistics Explore 2026 — DINOv3-B Standalone Severity Expert — Single GPU

This notebook is a **self-contained DINOv3-only pipeline** for the disaster classification competition.

Core recipe:

- backbone: `facebook/dinov3-vitb16-pretrain-lvd1689m`
- one GPU only
- full TRAIN directly; no fold loop
- EXIF-aware RGB decode, aspect-ratio-preserving resize + centered pad to `384×384`
- dense-aware feature: `CLS + masked mean of valid patch tokens`; register tokens are excluded
- disaster head + three disaster-conditioned severity heads
- auxiliary 9-class joint head as a low-weight regularizer
- loss: `0.15 disaster + 0.70 conditional severity + 0.15 joint9`
- severity/joint label smoothing: `0.03`
- exact cross-label conflicts are downweighted only; repeated same-label groups keep natural frequency
- Stage 1: backbone frozen, heads only, 2 epochs
- Stage 2: final 2 transformer blocks + final norm, 3 epochs
- snapshot probability blend: partial epochs 2 and 3
- no MixUp, CutMix, Random Erasing, aggressive crop, strong rotation, or TEST-derived calibration
- final output: one DINOv3-only `submission.csv`

The notebook never uses known TEST targets, filename ranges, TEST-to-TRAIN hash lookup, pseudo-labels, or probabilities from any previous model.


## Expected paths

```text
/workspace/dataset/SE/TRAIN
/workspace/dataset/SE/TEST
/workspace/dataset/SE/TRAIN/Solution.csv
```

DINOv3 checkpoints are gated. Before running, accept the DINOv3 model access conditions on Hugging Face and expose your token as the environment variable `HF_TOKEN`.


In [1]:
!nvidia-smi


Thu Sep  3 06:28:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   33C    P8              7W /  500W |     998MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Keep the Vast.ai PyTorch/CUDA build intact. Install only Python-side dependencies.
%pip install -q -U "transformers==4.57.1" "huggingface_hub>=0.34.0" pandas pillow tqdm safetensors


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os, math, random, time, json, hashlib
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor, get_cosine_schedule_with_warmup
from huggingface_hub import snapshot_download, get_token

TRAIN_DIR = Path('/workspace/dataset/SE/TRAIN')
TEST_DIR = Path('/workspace/dataset/SE/TEST')
SOLUTION_PATH = Path('/workspace/dataset/SE/TRAIN/Solution.csv')

OUTPUT_DIR = Path('/workspace/output/dinov3_b384_standalone_expert')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir(), f'Missing TRAIN_DIR: {TRAIN_DIR}'
assert TEST_DIR.is_dir(), f'Missing TEST_DIR: {TEST_DIR}'
assert SOLUTION_PATH.is_file(), f'Missing SOLUTION_PATH: {SOLUTION_PATH}'

# Reuse an existing environment/login token when available.
# Otherwise ask once with hidden input so the token is not stored in notebook text.
_existing_hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or get_token()
if not _existing_hf_token:
    _existing_hf_token = getpass('Paste Hugging Face READ token (hidden): ').strip()
    if _existing_hf_token:
        os.environ['HF_TOKEN'] = _existing_hf_token

print('TRAIN   :', TRAIN_DIR)
print('TEST    :', TEST_DIR)
print('SOLUTION:', SOLUTION_PATH)
print('OUTPUT  :', OUTPUT_DIR)
print('HF TOKEN:', 'SET' if (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or get_token()) else 'NOT SET')


Paste Hugging Face READ token (hidden):  ········


TRAIN   : /workspace/dataset/SE/TRAIN
TEST    : /workspace/dataset/SE/TEST
SOLUTION: /workspace/dataset/SE/TRAIN/Solution.csv
OUTPUT  : /workspace/output/dinov3_b384_standalone_expert
HF TOKEN: SET


## Single-GPU runtime policy

The statistical target keeps an effective batch size of 32. Micro-batch adapts to visible VRAM and gradient accumulation preserves the effective batch. BF16 is preferred when supported; otherwise FP16 is used.


In [4]:
assert torch.cuda.is_available(), 'A CUDA GPU is required.'
torch.cuda.set_device(0)
DEVICE = torch.device('cuda:0')
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / (1024**3)

if VRAM_GB >= 40:
    MICRO_BATCH = 32
elif VRAM_GB >= 22:
    MICRO_BATCH = 16
elif VRAM_GB >= 14:
    MICRO_BATCH = 8
else:
    MICRO_BATCH = 4

EFFECTIVE_BATCH = 32
assert EFFECTIVE_BATCH % MICRO_BATCH == 0
ACCUM_STEPS = EFFECTIVE_BATCH // MICRO_BATCH
EVAL_BATCH = min(64, max(8, MICRO_BATCH * 2))
NUM_WORKERS = min(12, max(4, (os.cpu_count() or 8)//4))
USE_BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# Throughput settings; they do not change architecture.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

print('torch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('GPU:', props.name)
print(f'VRAM: {VRAM_GB:.1f} GiB')
print('AMP:', 'BF16' if USE_BF16 else 'FP16')
print('micro batch:', MICRO_BATCH)
print('gradient accumulation:', ACCUM_STEPS)
print('effective batch:', EFFECTIVE_BATCH)
print('eval batch:', EVAL_BATCH)
print('workers:', NUM_WORKERS)


torch: 2.11.0+cu128
CUDA build: 12.8
GPU: NVIDIA GeForce RTX 5090
VRAM: 31.4 GiB
AMP: BF16
micro batch: 16
gradient accumulation: 2
effective batch: 32
eval batch: 32
workers: 4


## Model access and prefetch

This cell fails before any paid training if the Hugging Face token is absent or does not have access to the gated DINOv3 checkpoint. The downloaded snapshot is then loaded locally for the rest of the notebook.


In [5]:
MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or get_token()
if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN is not set. Accept the DINOv3 gated-model terms on Hugging Face, '
        'then export HF_TOKEN in the Vast.ai environment before running this cell.'
    )

try:
    MODEL_LOCAL_PATH = Path(snapshot_download(repo_id=MODEL_ID, token=HF_TOKEN))
except Exception as e:
    raise RuntimeError(
        'Could not download the gated DINOv3-B checkpoint. Confirm that the token has '
        'accepted access to facebook/dinov3-vitb16-pretrain-lvd1689m.'
    ) from e

print('DINOv3 snapshot:', MODEL_LOCAL_PATH)


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

DINOv3 snapshot: /root/.cache/huggingface/hub/models--facebook--dinov3-vitb16-pretrain-lvd1689m/snapshots/5931719e67bbdb9737e363e781fb0c67687896bc


## Fixed experiment configuration

The epoch counts and conservative partial fine-tuning are fixed before TEST inference. DINOv3-B is adapted lightly so the pretrained representation is preserved while the task-specific heads learn disaster and severity semantics.


In [6]:
SEED = 20260901
IMAGE_SIZE = 384
PATCH_SIZE = 16
PAD_RGB = (128,128,128)
DROPOUT = 0.15
UNFREEZE_LAST_N = 2

HEAD_EPOCHS = 2
PARTIAL_EPOCHS = 3
HEAD_LR = 5e-4
PARTIAL_HEAD_LR = 1e-4
PARTIAL_BACKBONE_LR = 5e-6
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.08
GRAD_CLIP = 1.0

W_DISASTER = 0.15
W_SEVERITY = 0.70
W_JOINT = 0.15
LABEL_SMOOTHING = 0.03
EXACT_CONFLICT_WEIGHT = 0.50

HFLIP_P = 0.50
BRIGHTNESS = 0.10
CONTRAST = 0.10
SATURATION = 0.05

IMAGE_EXTS = {'.jpg','.jpeg','.png','.jfif','.webp','.bmp','.tif','.tiff'}
JENIS = ['BANJIR','GEMPA BUMI','KEBAKARAN']
KERUSAKAN = ['KERUSAKAN RINGAN','KERUSAKAN SEDANG','KERUSAKAN BERAT']
JENIS_TO_IDX = {x:i for i,x in enumerate(JENIS)}
KER_TO_IDX = {x:i for i,x in enumerate(KERUSAKAN)}
IDX_TO_JENIS = {i:x for x,i in JENIS_TO_IDX.items()}
IDX_TO_KER = {i:x for x,i in KER_TO_IDX.items()}

# Explicit competition encoding. Never infer from TEST predictions.
SUB_JENIS = {'BANJIR':1, 'GEMPA BUMI':2, 'KEBAKARAN':3}
SUB_KER = {'KERUSAKAN BERAT':1, 'KERUSAKAN RINGAN':2, 'KERUSAKAN SEDANG':3}

MANIFEST_PATH = OUTPUT_DIR/'train_manifest_exact_conflicts.csv'
SNAP2 = OUTPUT_DIR/'partial_epoch_2.pt'
SNAP3 = OUTPUT_DIR/'partial_epoch_3.pt'
HISTORY_PATH = OUTPUT_DIR/'train_history.json'
DINO_PROB_PATH = OUTPUT_DIR/'test_probabilities_dinov3_snapshot23.npz'
DINO_SUBMISSION_PATH = OUTPUT_DIR/'submission.csv'

def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all()
print('Configuration ready.')


Configuration ready.


## TRAIN inventory and exact-conflict audit

Only exact SHA-256 groups with conflicting labels are downweighted. Same-label repeated groups retain their natural frequency. This targets irreducible annotation conflicts without changing the natural TRAIN distribution.


In [7]:
def enumerate_train(root):
    rows=[]
    for j_name in JENIS:
        jdir=root/j_name
        assert jdir.is_dir(), f'Missing disaster folder: {jdir}'
        for k_name in KERUSAKAN:
            kdir=jdir/k_name
            assert kdir.is_dir(), f'Missing severity folder: {kdir}'
            for p in kdir.iterdir():
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    rows.append({
                        'path':str(p),
                        'jenis_name':j_name,
                        'ker_name':k_name,
                        'jenis_idx':JENIS_TO_IDX[j_name],
                        'ker_idx':KER_TO_IDX[k_name],
                    })
    df=pd.DataFrame(rows).sort_values('path').reset_index(drop=True)
    df['joint_idx']=df.jenis_idx*3+df.ker_idx
    return df

def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

if MANIFEST_PATH.exists():
    manifest=pd.read_csv(MANIFEST_PATH)
    print('Loaded cached manifest:', MANIFEST_PATH)
else:
    manifest=enumerate_train(TRAIN_DIR)
    print('TRAIN images:',len(manifest))
    paths=manifest.path.tolist()
    hashes=[None]*len(paths)
    with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 8)) as ex:
        futures={ex.submit(sha256_file,p):i for i,p in enumerate(paths)}
        for fut in tqdm(futures,total=len(futures),desc='SHA256 TRAIN'):
            hashes[futures[fut]]=fut.result()
    manifest['sha256']=hashes

    sev_n=manifest.groupby('sha256').ker_idx.transform('nunique')
    dis_n=manifest.groupby('sha256').jenis_idx.transform('nunique')
    manifest['severity_conflict']=sev_n.gt(1)
    manifest['disaster_conflict']=dis_n.gt(1)
    manifest['severity_weight']=np.where(manifest.severity_conflict,EXACT_CONFLICT_WEIGHT,1.0).astype('float32')
    manifest['disaster_weight']=np.where(manifest.disaster_conflict,EXACT_CONFLICT_WEIGHT,1.0).astype('float32')
    manifest['joint_weight']=np.where(manifest.severity_conflict|manifest.disaster_conflict,EXACT_CONFLICT_WEIGHT,1.0).astype('float32')
    manifest.to_csv(MANIFEST_PATH,index=False)
    print('Saved:',MANIFEST_PATH)

assert len(manifest)>0 and manifest.path.is_unique
print('TRAIN images:',len(manifest))
print('Exact hash groups:',manifest.sha256.nunique())
print('Severity-conflict images:',int(manifest.severity_conflict.sum()))
print('Disaster-conflict images:',int(manifest.disaster_conflict.sum()))
print('\nJoint distribution:')
print(manifest.groupby(['jenis_name','ker_name']).size())


TRAIN images: 17482


SHA256 TRAIN:   0%|          | 0/17482 [00:00<?, ?it/s]

Saved: /workspace/output/dinov3_b384_standalone_expert/train_manifest_exact_conflicts.csv
TRAIN images: 17482
Exact hash groups: 15948
Severity-conflict images: 223
Disaster-conflict images: 2

Joint distribution:
jenis_name  ker_name        
BANJIR      KERUSAKAN BERAT     1968
            KERUSAKAN RINGAN    2018
            KERUSAKAN SEDANG    1971
GEMPA BUMI  KERUSAKAN BERAT     1623
            KERUSAKAN RINGAN    1393
            KERUSAKAN SEDANG    2730
KEBAKARAN   KERUSAKAN BERAT     2025
            KERUSAKAN RINGAN    1746
            KERUSAKAN SEDANG    2008
dtype: int64


## Preprocessing and dense-validity mask

The full scene is retained. Images are resized with aspect ratio preserved and centered on a 384×384 canvas. A 24×24 validity map tracks which DINO patch tokens correspond mainly to real image content, allowing the dense pooling branch to ignore padding.


In [8]:
processor = AutoImageProcessor.from_pretrained(str(MODEL_LOCAL_PATH), local_files_only=True)
IMAGE_MEAN = np.asarray(processor.image_mean,dtype=np.float32).reshape(3,1,1)
IMAGE_STD = np.asarray(processor.image_std,dtype=np.float32).reshape(3,1,1)
RESCALE_FACTOR = float(getattr(processor,'rescale_factor',1/255.0))
assert IMAGE_SIZE % PATCH_SIZE == 0
PATCH_GRID = IMAGE_SIZE//PATCH_SIZE
NUM_PATCHES = PATCH_GRID*PATCH_GRID


def decode_rgb(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im)
        if im.mode=='RGBA':
            bg=Image.new('RGBA',im.size,PAD_RGB+(255,))
            im=Image.alpha_composite(bg,im).convert('RGB')
        else:
            im=im.convert('RGB')
        return im.copy()


def mild_augment(im):
    if random.random()<HFLIP_P:
        im=ImageOps.mirror(im)
    if BRIGHTNESS>0:
        im=ImageEnhance.Brightness(im).enhance(1.0+random.uniform(-BRIGHTNESS,BRIGHTNESS))
    if CONTRAST>0:
        im=ImageEnhance.Contrast(im).enhance(1.0+random.uniform(-CONTRAST,CONTRAST))
    if SATURATION>0:
        im=ImageEnhance.Color(im).enhance(1.0+random.uniform(-SATURATION,SATURATION))
    return im


def letterbox_and_patch_mask(im):
    w,h=im.size
    scale=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
    nw=max(1,int(round(w*scale))); nh=max(1,int(round(h*scale)))
    im=im.resize((nw,nh),Image.Resampling.BICUBIC)
    x0=(IMAGE_SIZE-nw)//2; y0=(IMAGE_SIZE-nh)//2
    canvas=Image.new('RGB',(IMAGE_SIZE,IMAGE_SIZE),PAD_RGB)
    canvas.paste(im,(x0,y0))

    pixel_mask=np.zeros((IMAGE_SIZE,IMAGE_SIZE),dtype=np.float32)
    pixel_mask[y0:y0+nh,x0:x0+nw]=1.0
    patch_fraction=pixel_mask.reshape(PATCH_GRID,PATCH_SIZE,PATCH_GRID,PATCH_SIZE).mean(axis=(1,3))
    patch_mask=(patch_fraction>=0.50).astype(np.float32).reshape(-1)
    if patch_mask.sum()<1:
        patch_mask[np.argmax(patch_fraction.reshape(-1))]=1.0
    return canvas,patch_mask


def to_normalized_tensor(im):
    arr=np.asarray(im,dtype=np.float32).transpose(2,0,1)
    arr=arr*RESCALE_FACTOR
    arr=(arr-IMAGE_MEAN)/IMAGE_STD
    return torch.from_numpy(arr)

class TrainDS(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        im=mild_augment(decode_rgb(r.path))
        canvas,pm=letterbox_and_patch_mask(im)
        return {
            'pixel_values':to_normalized_tensor(canvas),
            'patch_mask':torch.from_numpy(pm),
            'jenis':torch.tensor(int(r.jenis_idx),dtype=torch.long),
            'ker':torch.tensor(int(r.ker_idx),dtype=torch.long),
            'joint':torch.tensor(int(r.joint_idx),dtype=torch.long),
            'severity_weight':torch.tensor(float(r.severity_weight),dtype=torch.float32),
            'disaster_weight':torch.tensor(float(r.disaster_weight),dtype=torch.float32),
            'joint_weight':torch.tensor(float(r.joint_weight),dtype=torch.float32),
        }

class TestDS(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        canvas,pm=letterbox_and_patch_mask(decode_rgb(r.path))
        return {'pixel_values':to_normalized_tensor(canvas),'patch_mask':torch.from_numpy(pm)}

# Lightweight preprocessing test.
_sample=manifest.iloc[0]
_item=TrainDS(manifest.iloc[:1])[0]
assert _item['pixel_values'].shape==(3,IMAGE_SIZE,IMAGE_SIZE)
assert _item['patch_mask'].shape==(NUM_PATCHES,)
assert float(_item['patch_mask'].sum())>0
print('Preprocessing OK | valid patches:',int(_item['patch_mask'].sum()),'/',NUM_PATCHES)


Preprocessing OK | valid patches: 432 / 576


## DINOv3 standalone expert

The DINOv3 sequence is split into CLS, register, and patch tokens. Register tokens are excluded from local pooling. The final feature concatenates the CLS representation with a masked mean over valid image patches.

The helper below discovers encoder blocks robustly across compatible `AutoModel` wrappers, so partial fine-tuning does not depend on a single hard-coded module path.


In [9]:
def _get_module_by_path(root, path):
    obj = root
    for part in path.split('.'):
        if not hasattr(obj, part):
            raise AttributeError(path)
        obj = getattr(obj, part)
    return obj


def get_dino_layers(backbone):
    expected = int(getattr(backbone.config, 'num_hidden_layers', 0))
    candidates = [
        'model.layer',
        'model.model.layer',
        'encoder.layer',
        'model.encoder.layer',
        'encoder.layers',
        'model.encoder.layers',
        'layers',
        'model.layers',
    ]

    for path in candidates:
        try:
            layers = _get_module_by_path(backbone, path)
        except AttributeError:
            continue
        if isinstance(layers, (nn.ModuleList, list, tuple)) and len(layers) > 0:
            if expected == 0 or len(layers) == expected:
                return layers

    # Robust fallback: find the ModuleList matching config.num_hidden_layers.
    fallback = []
    for name, module in backbone.named_modules():
        if isinstance(module, nn.ModuleList) and len(module) > 0:
            if expected == 0 or len(module) == expected:
                fallback.append((name, module))

    if len(fallback) == 1:
        return fallback[0][1]

    if fallback:
        # Prefer a path whose name looks like an encoder/block stack.
        for name, module in fallback:
            lname = name.lower()
            if any(key in lname for key in ('layer', 'encoder', 'block')):
                return module

    top = list(dict(backbone.named_children()).keys())
    found = [(n, len(m)) for n, m in backbone.named_modules()
             if isinstance(m, nn.ModuleList)]
    raise AttributeError(
        'Could not locate DINOv3 transformer layers. '
        f'Backbone={type(backbone).__name__}, top-level={top}, ModuleLists={found}'
    )


def get_dino_final_norm(backbone):
    for path in [
        'norm',
        'model.norm',
        'layernorm',
        'model.layernorm',
        'final_layer_norm',
        'model.final_layer_norm',
    ]:
        try:
            module = _get_module_by_path(backbone, path)
        except AttributeError:
            continue
        if isinstance(module, nn.Module):
            return module
    return None


def disable_dino_position_augmentation(backbone):
    # DINOv3 RoPE positional augmentation should stay deterministic here.
    for path in ['rope_embeddings', 'model.rope_embeddings']:
        try:
            module = _get_module_by_path(backbone, path)
        except AttributeError:
            continue
        if isinstance(module, nn.Module):
            module.eval()


class DINOv3StandaloneExpert(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            str(MODEL_LOCAL_PATH),
            local_files_only=True,
        )
        h = int(self.backbone.config.hidden_size)
        self.num_register_tokens = int(
            getattr(self.backbone.config, 'num_register_tokens', 0)
        )
        self.feature_norm = nn.LayerNorm(h * 2)
        self.dropout = nn.Dropout(DROPOUT)
        self.disaster = nn.Linear(h * 2, 3)
        self.severity = nn.ModuleList([
            nn.Linear(h * 2, 3) for _ in range(3)
        ])
        self.joint = nn.Linear(h * 2, 9)

    def features(self, x, patch_mask):
        out = self.backbone(pixel_values=x)
        hs = out.last_hidden_state

        cls = hs[:, 0, :]
        patches = hs[:, 1 + self.num_register_tokens:, :]

        if patches.shape[1] != patch_mask.shape[1]:
            raise RuntimeError(
                'Patch-token mismatch: '
                f'model={patches.shape[1]}, mask={patch_mask.shape[1]}, '
                f'register_tokens={self.num_register_tokens}'
            )

        w = patch_mask.to(dtype=patches.dtype).unsqueeze(-1)
        patch_mean = (patches * w).sum(1) / w.sum(1).clamp_min(1.0)

        z = torch.cat([cls, patch_mean], dim=-1)
        return self.dropout(self.feature_norm(z))

    def forward(self, x, patch_mask):
        z = self.features(x, patch_mask)
        j = self.disaster(z)
        s = torch.stack([head(z) for head in self.severity], dim=1)
        q = self.joint(z)
        return j, s, q

    def probabilities(self, x, patch_mask):
        j, s, _ = self.forward(x, patch_mask)
        jp = F.softmax(j.float(), dim=-1)
        sp = F.softmax(s.float(), dim=-1)
        kp = (jp.unsqueeze(-1) * sp).sum(dim=1)
        jp = jp / jp.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        kp = kp / kp.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        return jp, kp


def set_trainable(model, stage):
    for p in model.backbone.parameters():
        p.requires_grad = False

    for module in [
        model.feature_norm,
        model.disaster,
        model.severity,
        model.joint,
    ]:
        for p in module.parameters():
            p.requires_grad = True

    if stage == 'partial':
        layers = get_dino_layers(model.backbone)
        assert len(layers) >= UNFREEZE_LAST_N, (
            f'Only {len(layers)} blocks found, requested {UNFREEZE_LAST_N}.'
        )
        for layer in layers[-UNFREEZE_LAST_N:]:
            for p in layer.parameters():
                p.requires_grad = True

        final_norm = get_dino_final_norm(model.backbone)
        if final_norm is not None:
            for p in final_norm.parameters():
                p.requires_grad = True


def weighted_mean(loss, w):
    return (loss * w).sum() / w.sum().clamp_min(1e-8)


def compute_loss(model, b):
    x = b['pixel_values'].to(DEVICE, non_blocking=True)
    pm = b['patch_mask'].to(DEVICE, non_blocking=True)
    yj = b['jenis'].to(DEVICE, non_blocking=True)
    yk = b['ker'].to(DEVICE, non_blocking=True)
    yq = b['joint'].to(DEVICE, non_blocking=True)
    sw = b['severity_weight'].to(DEVICE, non_blocking=True)
    dw = b['disaster_weight'].to(DEVICE, non_blocking=True)
    qw = b['joint_weight'].to(DEVICE, non_blocking=True)

    j, s, q = model(x, pm)
    row = torch.arange(x.shape[0], device=DEVICE)
    conditional = s[row, yj]

    lj = F.cross_entropy(j, yj, reduction='none')
    lk = F.cross_entropy(
        conditional,
        yk,
        reduction='none',
        label_smoothing=LABEL_SMOOTHING,
    )
    lq = F.cross_entropy(
        q,
        yq,
        reduction='none',
        label_smoothing=LABEL_SMOOTHING,
    )

    lj = weighted_mean(lj, dw)
    lk = weighted_mean(lk, sw)
    lq = weighted_mean(lq, qw)

    total = W_DISASTER * lj + W_SEVERITY * lk + W_JOINT * lq
    return total, {
        'disaster': float(lj.detach()),
        'severity': float(lk.detach()),
        'joint': float(lq.detach()),
    }


## Model-loader smoke test

This catches gated-checkpoint, Transformers API, token-layout, and 384-patch-mask mismatches before the paid training loop starts.


In [10]:
seed_all()

model = DINOv3StandaloneExpert().to(DEVICE)

# Heads-only configuration.
set_trainable(model, 'heads')
heads_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

model.eval()
x = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
pm = torch.ones(1, NUM_PATCHES, device=DEVICE)

with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DTYPE):
    j, s, q = model(x, pm)

assert j.shape == (1, 3)
assert s.shape == (1, 3, 3)
assert q.shape == (1, 9)

layers = get_dino_layers(model.backbone)
final_norm = get_dino_final_norm(model.backbone)

expected_layers = int(model.backbone.config.num_hidden_layers)
assert len(layers) == expected_layers, (
    f'Expected {expected_layers} DINO blocks, found {len(layers)}.'
)

# Verify partial fine-tuning really opens the intended backbone blocks.
set_trainable(model, 'partial')
partial_trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
assert partial_trainable > heads_trainable
assert all(
    p.requires_grad
    for layer in layers[-UNFREEZE_LAST_N:]
    for p in layer.parameters()
), 'The selected final DINO blocks were not fully unfrozen.'

print('SMOKE OK')
print('backbone class:', type(model.backbone).__name__)
print('hidden size:', model.backbone.config.hidden_size)
print('register tokens:', model.num_register_tokens)
print('vision blocks:', len(layers))
print('final norm found:', final_norm is not None)
print('patch tokens at 384:', NUM_PATCHES)
print(f'heads-only trainable params: {heads_trainable/1e6:.2f}M')
print(f'partial-FT trainable params: {partial_trainable/1e6:.2f}M')

del model, x, pm, j, s, q
torch.cuda.empty_cache()


SMOKE OK
backbone class: DINOv3ViTModel
hidden size: 768
register tokens: 4
vision blocks: 12
final norm found: True
patch tokens at 384: 576
heads-only trainable params: 0.04M
partial-FT trainable params: 14.21M


## Full-TRAIN training

The run trains all labelled TRAIN images directly. No TEST information is used for model selection.

- heads-only: 2 epochs
- partial FT: 3 epochs, final 2 DINO blocks only
- save partial epochs 2 and 3
- final DINO probabilities are the 50/50 snapshot blend


In [11]:
def make_optimizer(model,stage,epochs,loader_len):
    if stage=='heads':
        params=[p for p in model.parameters() if p.requires_grad]
        opt=torch.optim.AdamW(params,lr=HEAD_LR,weight_decay=WEIGHT_DECAY)
    else:
        backbone=[]; heads=[]
        for name,p in model.named_parameters():
            if not p.requires_grad: continue
            if name.startswith('backbone.'):
                backbone.append(p)
            else:
                heads.append(p)
        opt=torch.optim.AdamW([
            {'params':backbone,'lr':PARTIAL_BACKBONE_LR},
            {'params':heads,'lr':PARTIAL_HEAD_LR},
        ],weight_decay=WEIGHT_DECAY)
    steps_per_epoch=math.ceil(loader_len/ACCUM_STEPS)
    total_steps=max(1,steps_per_epoch*epochs)
    warmup=max(1,int(total_steps*WARMUP_RATIO))
    sch=get_cosine_schedule_with_warmup(opt,warmup,total_steps)
    return opt,sch


def train_stage(model,loader,stage,epochs,history):
    set_trainable(model,stage)
    model.train()
    # Frozen-head stage uses deterministic pretrained features. During partial FT,
    # keep DINO's RoPE coordinate augmentation disabled so the experiment changes
    # representation weights, not spatial-position policy.
    if stage=='heads':
        model.backbone.eval()
    else:
        disable_dino_position_augmentation(model.backbone)
    opt,sch=make_optimizer(model,stage,epochs,len(loader))
    scaler=torch.cuda.amp.GradScaler(enabled=(AMP_DTYPE==torch.float16))

    for ep in range(1,epochs+1):
        t0=time.time(); opt.zero_grad(set_to_none=True)
        loss_sum=0.0; n=0
        comp={'disaster':0.0,'severity':0.0,'joint':0.0}
        pbar=tqdm(enumerate(loader),total=len(loader),desc=f'{stage} {ep}/{epochs}')
        for step,b in pbar:
            with torch.autocast('cuda',dtype=AMP_DTYPE):
                loss,parts=compute_loss(model,b)
                scaled_loss=loss/ACCUM_STEPS
            scaler.scale(scaled_loss).backward()
            do_step=((step+1)%ACCUM_STEPS==0) or (step+1==len(loader))
            if do_step:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()
            bs=b['jenis'].shape[0]
            loss_sum+=float(loss.detach())*bs; n+=bs
            for k in comp: comp[k]+=parts[k]*bs
            pbar.set_postfix(loss=f'{loss_sum/max(1,n):.4f}')

        rec={'stage':stage,'epoch':ep,'loss':loss_sum/n,
             **{f'{k}_loss':v/n for k,v in comp.items()},
             'minutes':(time.time()-t0)/60}
        history.append(rec)
        print(json.dumps(rec,indent=2))
        HISTORY_PATH.write_text(json.dumps(history,indent=2))

        if stage=='partial' and ep in {2,3}:
            path=SNAP2 if ep==2 else SNAP3
            torch.save({'model':model.state_dict(),'stage':stage,'epoch':ep,'config':{
                'model_id':MODEL_ID,'image_size':IMAGE_SIZE,'unfreeze_last_n':UNFREEZE_LAST_N,
                'loss_weights':[W_DISASTER,W_SEVERITY,W_JOINT],
            }},path)
            print('Saved snapshot:',path)

seed_all()
if SNAP2.exists() and SNAP3.exists():
    print('Both DINO snapshots already exist; skipping paid training.')
    print(SNAP2); print(SNAP3)
else:
    train_ds=TrainDS(manifest)
    g=torch.Generator(); g.manual_seed(SEED)
    train_loader=DataLoader(
        train_ds,batch_size=MICRO_BATCH,shuffle=True,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0),drop_last=False,generator=g,
    )
    model=DINOv3StandaloneExpert().to(DEVICE)
    history=[]
    train_stage(model,train_loader,'heads',HEAD_EPOCHS,history)
    train_stage(model,train_loader,'partial',PARTIAL_EPOCHS,history)
    del model,train_loader,train_ds
    torch.cuda.empty_cache()

assert SNAP2.exists() and SNAP3.exists(), 'Training did not produce both selected snapshots.'
print('Training artifacts ready.')


/tmp/ipykernel_1721/3992291563.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=(AMP_DTYPE==torch.float16))


heads 1/2:   0%|          | 0/1093 [00:00<?, ?it/s]

{
  "stage": "heads",
  "epoch": 1,
  "loss": 0.5666191497788442,
  "disaster_loss": 0.11549895334232992,
  "severity_loss": 0.6079804260279873,
  "joint_loss": 0.8247200683937262,
  "minutes": 1.8246795018513997
}


heads 2/2:   0%|          | 0/1093 [00:00<?, ?it/s]

{
  "stage": "heads",
  "epoch": 2,
  "loss": 0.4226481838386428,
  "disaster_loss": 0.023093850926634917,
  "severity_loss": 0.46988019627791605,
  "joint_loss": 0.6017864650494037,
  "minutes": 1.785916233062744
}


...

{
  "stage": "partial",
  "epoch": 1,
  "loss": 0.40752401725335685,
  "disaster_loss": 0.020154971630766284,
  "severity_loss": 0.45312132368697655,
  "joint_loss": 0.5821056431497129,
  "minutes": 1.8194225231806438
}


partial 2/3:   0%|          | 0/1093 [00:00<?, ?it/s]

{
  "stage": "partial",
  "epoch": 2,
  "loss": 0.38351286544214463,
  "disaster_loss": 0.017406140910771594,
  "severity_loss": 0.42545883719100924,
  "joint_loss": 0.5538717281039663,
  "minutes": 1.7818743149439493
}
Saved snapshot: /workspace/output/dinov3_b384_standalone_expert/partial_epoch_2.pt


partial 3/3:   0%|          | 0/1093 [00:00<?, ?it/s]

{
  "stage": "partial",
  "epoch": 3,
  "loss": 0.36906161864784975,
  "disaster_loss": 0.01580209444115997,
  "severity_loss": 0.40901023951742554,
  "joint_loss": 0.53589424973371,
  "minutes": 1.7728933850924173
}
Saved snapshot: /workspace/output/dinov3_b384_standalone_expert/partial_epoch_3.pt
Training artifacts ready.


## TEST inference: DINO snapshot blend

TEST inference is deterministic and centered. Probabilities are recomputed in FP32 and normalized after averaging partial epochs 2 and 3.


In [12]:
def enumerate_test(root):
    rows=[]
    for p in root.iterdir():
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rows.append({'path':str(p),'id':str(p.stem)})
    def sort_key(r):
        return (0,int(r['id'])) if r['id'].isdigit() else (1,r['id'])
    df=pd.DataFrame(sorted(rows,key=sort_key))
    assert df.id.is_unique, 'TEST stems must be unique.'
    return df

@torch.no_grad()
def predict_snapshot(path,loader):
    model=DINOv3StandaloneExpert().to(DEVICE)
    ck=torch.load(path,map_location='cpu')
    model.load_state_dict(ck['model'],strict=True)
    model.eval(); jp=[]; kp=[]
    for b in tqdm(loader,desc=f'Infer {Path(path).name}',leave=False):
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        pm=b['patch_mask'].to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            j,s,_=model(x,pm)
        jprob=F.softmax(j.float(),dim=-1)
        sprob=F.softmax(s.float(),dim=-1)
        kprob=(jprob.unsqueeze(-1)*sprob).sum(1)
        jprob=jprob/jprob.sum(1,keepdim=True).clamp_min(1e-12)
        kprob=kprob/kprob.sum(1,keepdim=True).clamp_min(1e-12)
        jp.append(jprob.cpu().numpy()); kp.append(kprob.cpu().numpy())
    del model; torch.cuda.empty_cache()
    return np.concatenate(jp).astype(np.float32),np.concatenate(kp).astype(np.float32)

test_df=enumerate_test(TEST_DIR)
print('TEST images:',len(test_df))
test_loader=DataLoader(TestDS(test_df),batch_size=EVAL_BATCH,shuffle=False,num_workers=NUM_WORKERS,
                       pin_memory=True,persistent_workers=(NUM_WORKERS>0))

j2,k2=predict_snapshot(SNAP2,test_loader)
j3,k3=predict_snapshot(SNAP3,test_loader)
dino_j=(j2+j3)/2.0; dino_k=(k2+k3)/2.0
dino_j=dino_j/np.clip(dino_j.sum(1,keepdims=True),1e-12,None)
dino_k=dino_k/np.clip(dino_k.sum(1,keepdims=True),1e-12,None)
assert np.isfinite(dino_j).all() and np.isfinite(dino_k).all()
assert np.allclose(dino_j.sum(1),1,atol=1e-6)
assert np.allclose(dino_k.sum(1),1,atol=1e-6)

np.savez_compressed(DINO_PROB_PATH,
    ids=test_df.id.astype(str).to_numpy(),
    jenis_prob=dino_j,kerusakan_prob=dino_k,
    jenis_prob_epoch2=j2,kerusakan_prob_epoch2=k2,
    jenis_prob_epoch3=j3,kerusakan_prob_epoch3=k3,
)
print('Saved:',DINO_PROB_PATH)


TEST images: 450


Infer partial_epoch_2.pt:   0%|          | 0/15 [00:00<?, ?it/s]

Infer partial_epoch_3.pt:   0%|          | 0/15 [00:00<?, ?it/s]

Saved: /workspace/output/dinov3_b384_standalone_expert/test_probabilities_dinov3_snapshot23.npz


## Final DINOv3-only submission

The final prediction is the 50/50 probability blend of partial epochs 2 and 3 from this same DINOv3 training run. No probabilities from any other notebook or model are used.


In [13]:
def build_submission(jprob,kprob,path):
    pred_j=jprob.argmax(1); pred_k=kprob.argmax(1)
    by_id={
        str(test_df.iloc[i].id):(
            SUB_JENIS[IDX_TO_JENIS[int(pred_j[i])]],
            SUB_KER[IDX_TO_KER[int(pred_k[i])]],
        ) for i in range(len(test_df))
    }
    solution=pd.read_csv(SOLUTION_PATH,sep=None,engine='python')
    assert solution.columns.tolist()==['ID','Target']
    out=solution.copy(); targets=[]
    for rid in out.ID.astype(str):
        base,suffix=rid.rsplit('_',1)
        assert base in by_id, f'Missing TEST ID: {base}'
        if suffix=='jenis': targets.append(by_id[base][0])
        elif suffix=='kerusakan': targets.append(by_id[base][1])
        else: raise ValueError(f'Unexpected solution suffix: {suffix}')
    out['Target']=pd.Series(targets,dtype='int64')
    out.to_csv(path,index=False)
    chk=pd.read_csv(path)
    assert chk.columns.tolist()==['ID','Target']
    assert len(chk)==len(solution)==2*len(test_df)
    assert chk.ID.astype(str).tolist()==solution.ID.astype(str).tolist()
    assert chk.Target.notna().all()
    assert set(chk.Target.astype(int).unique()).issubset({1,2,3})
    return out

dino_submission=build_submission(dino_j,dino_k,DINO_SUBMISSION_PATH)
print(dino_submission.head(12).to_string(index=False))
print('FINAL submission:', DINO_SUBMISSION_PATH)


         ID  Target
    1_jenis       1
1_kerusakan       2
    2_jenis       1
2_kerusakan       1
    3_jenis       1
3_kerusakan       1
    4_jenis       1
4_kerusakan       2
    5_jenis       1
5_kerusakan       1
    6_jenis       1
6_kerusakan       2
FINAL submission: /workspace/output/dinov3_b384_standalone_expert/submission.csv


## Final artifact checks

The model was trained only on labelled TRAIN images. TEST is used only for deterministic inference. The submission uses only the DINOv3 snapshots trained in this notebook, preserves sample-solution row order, and uses explicit numeric encodings.


In [14]:
assert DINO_SUBMISSION_PATH.exists()

submission = pd.read_csv(DINO_SUBMISSION_PATH)
assert submission.shape[0] == 900
assert submission.columns.tolist() == ['ID', 'Target']
assert set(submission.Target.astype(int).unique()).issubset({1, 2, 3})

summary = {
    'backbone': MODEL_ID,
    'input_size': IMAGE_SIZE,
    'single_gpu': True,
    'effective_batch': EFFECTIVE_BATCH,
    'head_epochs': HEAD_EPOCHS,
    'partial_epochs': PARTIAL_EPOCHS,
    'unfrozen_last_blocks': UNFREEZE_LAST_N,
    'feature_pooling': 'CLS + masked valid-patch mean',
    'loss_weights': {
        'disaster': W_DISASTER,
        'severity': W_SEVERITY,
        'joint9': W_JOINT,
    },
    'label_smoothing': LABEL_SMOOTHING,
    'snapshot_blend': ['partial_epoch_2', 'partial_epoch_3'],
    'probabilities': str(DINO_PROB_PATH),
    'submission': str(DINO_SUBMISSION_PATH),
}

(OUTPUT_DIR / 'run_summary.json').write_text(
    json.dumps(summary, indent=2)
)

print(json.dumps(summary, indent=2))
print('\nREADY TO SUBMIT:', DINO_SUBMISSION_PATH)


{
  "backbone": "facebook/dinov3-vitb16-pretrain-lvd1689m",
  "input_size": 384,
  "single_gpu": true,
  "effective_batch": 32,
  "head_epochs": 2,
  "partial_epochs": 3,
  "unfrozen_last_blocks": 2,
  "feature_pooling": "CLS + masked valid-patch mean",
  "loss_weights": {
    "disaster": 0.15,
    "severity": 0.7,
    "joint9": 0.15
  },
  "label_smoothing": 0.03,
  "snapshot_blend": [
    "partial_epoch_2",
    "partial_epoch_3"
  ],
  "probabilities": "/workspace/output/dinov3_b384_standalone_expert/test_probabilities_dinov3_snapshot23.npz",
  "submission": "/workspace/output/dinov3_b384_standalone_expert/submission.csv"
}

READY TO SUBMIT: /workspace/output/dinov3_b384_standalone_expert/submission.csv
